In [35]:
#import necessary libararies
from pypdf import PdfReader
from pathlib import Path
import glob
import numpy as np
import os
from sentence_transformers import SentenceTransformer

In [6]:
folder_path = r"C:\Users\Sumed\Desktop\rag_assistant\data\*pdf"
pdf_files = glob.glob(folder_path)

if not pdf_files:
        raise Exception('No document(s) found')
else:
    print(f'{len(pdf_files)} document(s) found')

for i in pdf_files:
    print(i)

2 document(s) found
C:\Users\Sumed\Desktop\rag_assistant\data\encoder_decoder_neural_networks.pdf
C:\Users\Sumed\Desktop\rag_assistant\data\Mathematics of Deep Learning.pdf


In [ ]:
#some insights about the PDFs in the folder

page_counter = 0
total_words = 0
words_per_page = []
average_words_per_page = 0

for i in pdf_files:
    print('processing file : ', i)
    reader = PdfReader(i)
    max_words_per_page = 0

    for _,j in enumerate(reader.pages):
        current_page = j.extract_text()
        total_words_in_page = current_page.split()
        
        if max_words_per_page < len(total_words_in_page):
            max_words_per_page = len(total_words_in_page)
        words_per_page.append(len(total_words_in_page))

    print('max words in a page :',max_words_per_page) 
    print('total words in the PDF :',sum(words_per_page))
    print('average number of words per page: ', round(np.divide( sum(words_per_page),len(words_per_page) ),0) )
    print('\n')

processing file :  C:\Users\Sumed\Desktop\rag_assistant\data\encoder_decoder_neural_networks.pdf
max words in a page : 1683
total words in the PDF : 49540
average number of words per page:  266.0


processing file :  C:\Users\Sumed\Desktop\rag_assistant\data\Mathematics of Deep Learning.pdf
max words in a page : 596
total words in the PDF : 95328
average number of words per page:  298.0




In [32]:
# create chunks
all_chunks = []
chunks = {}
for i in pdf_files:
    print('processing file : ', i)
    reader = PdfReader(i)
    file_name = os.path.basename(i)
    page_counter = 0

    for _,j in enumerate(reader.pages):
        current_page = j.extract_text()
        chunks = {'source_file': file_name
                 ,'page_number': page_counter
                 ,'chunk_id'   : f'{file_name}_{page_counter}'
                 ,'text'       : current_page
        }
        page_counter+=1
        all_chunks.append(chunks)
    
print(f'sample chunks : {all_chunks[:5]}')
        

processing file :  C:\Users\Sumed\Desktop\rag_assistant\data\encoder_decoder_neural_networks.pdf
processing file :  C:\Users\Sumed\Desktop\rag_assistant\data\Mathematics of Deep Learning.pdf
sample chunks : [{'source_file': 'encoder_decoder_neural_networks.pdf', 'page_number': 0, 'chunk_id': 'encoder_decoder_neural_networks.pdf_0', 'text': 'ENCODER-DECODER\nNEURAL NETWORKS\nNal Kalchbrenner'}, {'source_file': 'encoder_decoder_neural_networks.pdf', 'page_number': 1, 'chunk_id': 'encoder_decoder_neural_networks.pdf_1', 'text': 'Encoder-Decoder Neural Networks\nNal Kalchbrenner\nNew College\nUniversity of Oxford\nA thesis submitted for the degree of\nDoctor of Philosophy\nMichaelmas 2017'}, {'source_file': 'encoder_decoder_neural_networks.pdf', 'page_number': 2, 'chunk_id': 'encoder_decoder_neural_networks.pdf_2', 'text': 'To my parents Metka and Bojan'}, {'source_file': 'encoder_decoder_neural_networks.pdf', 'page_number': 3, 'chunk_id': 'encoder_decoder_neural_networks.pdf_3', 'text': '

In [36]:
#create embeddings
embedding_model = SentenceTransformer('BAAI/bge-small-en-v1.5')
embeddings = embedding_model.encode(all_chunks)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

c:\Users\Sumed\AppData\Local\Programs\Python\Python313\Lib\site-packages\huggingface_hub\file_download.py:137: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Sumed\.cache\huggingface\hub\models--BAAI--bge-small-en-v1.5. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/94.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

ValueError: Multimodal dict input contains unrecognized modality keys: {'chunk_id', 'source_file', 'page_number'}. Expected keys from: ['audio', 'image', 'text', 'video']